# 第10章 ニューラルネットワークと万能近似定理 ― デモノートブック

万能近似定理（10.4 節）は「1 隠れ層で連続関数を一様に近似できる」という**存在定理**である。
このノートブックでは、その構成的証明（10.4.2 節）を実際に組み立てて誤差の収束次数を測り、
定理が言っていないこと（幅・学習可能性）を数値で確かめる。
あわせて活性化関数（図 10.1）、ReLU 網の区分線形表現（式 10.23）、深さの効果（10.6 節）、
誤差逆伝播（式 10.32、式 10.33）、XOR（10.3 節）、Barron クラス（10.7 節）を扱う。

## 目次

- 10.1 活性化関数とその導関数（10.2 節、図 10.1）
- 10.2 万能近似定理の構成的証明 ― hat 関数と 1 の分割（10.4.2 節、式 10.13〜式 10.17）
- 10.3 近似誤差の収束次数 $O(N^{-2})$（定理「1 次元 $C^2$ 関数の近似率」、式 10.24）
- 10.4 折れ目の数と深さの効果（10.5〜10.6 節、式 10.23、図 10.4）
- 10.5 誤差逆伝播法のスクラッチ実装と数値微分による検証（式 10.32、式 10.33）
- 10.6 XOR ― 1 隠れ層 2 ユニットで解ける（10.3 節、式 10.2）
- 10.7 学習の実験と Barron クラス（10.7 節、式 10.26）
- 演習 / 演習の解答

行列の規約：**データ行列は列がサンプル**（$\boldsymbol{X}\in\mathbb{R}^{d\times n}$）である。
scikit-learn は行がサンプルなので、渡すときに `X.T` と転置する。
PyTorch は使わず numpy と scikit-learn だけで書く。

## 準備

最初にこのセルを実行する。日本語フォントの設定（Colab には既定で入っていない）と、
以降で使うライブラリの読み込みを行う。フォントの導入に失敗した場合は
図のラベルが自動的に英語に切り替わる（`L()` 関数）。

In [ ]:
import subprocess, sys, warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings("ignore", category=UserWarning)


def _setup_japanese_font():
    """日本語が出せるフォントを探し、なければ入れる。成功したら True。"""
    cands = ["IPAexGothic", "IPAGothic", "Noto Sans CJK JP", "Noto Sans JP",
             "TakaoGothic", "Yu Gothic", "Hiragino Sans"]
    have = {f.name for f in fm.fontManager.ttflist}
    for name in cands:
        if name in have:
            matplotlib.rcParams["font.family"] = name
            return True
    # Colab 想定：pip で導入する
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "japanize-matplotlib"], check=True, timeout=180)
        import japanize_matplotlib  # noqa: F401  読み込むだけで設定される
        return True
    except Exception:
        pass
    # 予備：apt で IPA フォント
    try:
        subprocess.run("apt-get -qq -y install fonts-ipafont-gothic",
                       shell=True, check=True, timeout=300)
        fm._load_fontmanager(try_read_cache=False)
        matplotlib.rcParams["font.family"] = "IPAGothic"
        return True
    except Exception:
        return False


JP = _setup_japanese_font()


def L(ja, en):
    """日本語フォントが使えれば ja、駄目なら en を返す（図のラベル用）。"""
    return ja if JP else en


matplotlib.rcParams.update({
    "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "figure.dpi": 110, "savefig.bbox": "tight",
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.unicode_minus": False,
})

# 講義ノートの図と同じ色
C = {"blue": "#1f4e79", "red": "#c0392b", "green": "#1e8449",
     "orange": "#d68910", "purple": "#6a4c93", "gray": "#7f8c8d"}

print("日本語フォント:", "有効" if JP else "無効（図のラベルは英語になる）")
print("numpy", np.__version__, "| matplotlib", matplotlib.__version__)

## 10.1 活性化関数とその導関数

図 10.1 に対応する。シグモイド $\sigma(t)=1/(1+e^{-t})$、$\tanh$、
ReLU $\max(t,0)$、GELU $t\Phi(t)$ とその導関数を並べる。
講義ノートの記述——シグモイドの導関数の最大値は $1/4$、$\tanh$ のそれは $1$、
ReLU の導関数は階段関数、GELU は $t\approx-0.75$ で最小値 $-0.17$ をとり
導関数も負（最小 $-0.13$）になる——を数値で確かめる。

In [ ]:
from scipy.stats import norm

t = np.linspace(-4, 4, 2001)
sig = 1 / (1 + np.exp(-t))
acts = {
    L("シグモイド", "sigmoid"): (sig, sig * (1 - sig), C["blue"], "-"),
    "tanh":                     (np.tanh(t), 1 - np.tanh(t) ** 2, C["red"], "--"),
    "ReLU":                     (np.maximum(t, 0), (t > 0).astype(float), C["green"], "-"),
    "GELU":                     (t * norm.cdf(t), norm.cdf(t) + t * norm.pdf(t),
                                 C["purple"], "--"),
}
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
for name, (f_, df_, col, ls) in acts.items():
    axes[0].plot(t, f_, color=col, ls=ls, lw=1.8, label=name)
    axes[1].plot(t, df_, color=col, ls=ls, lw=1.8, label=name)
axes[1].axhline(0.25, color=C["gray"], ls=":", lw=1.0)
axes[1].axhline(0.0, color="k", lw=0.8)
axes[0].set_title(L("活性化関数", "activation functions"))
axes[1].set_title(L("その導関数", "their derivatives"))
for ax in axes:
    ax.set_xlabel("$t$")
    ax.legend(fontsize=9)
axes[0].set_ylabel(r"$\sigma(t)$"); axes[1].set_ylabel(r"$\sigma'(t)$")
plt.show()

gelu, dgelu = acts["GELU"][0], acts["GELU"][1]
print("シグモイド導関数の最大値 =", round(float((sig * (1 - sig)).max()), 4),
      " (t =", round(float(t[(sig * (1 - sig)).argmax()]), 2), ")")
print("tanh 導関数の最大値      =", round(float((1 - np.tanh(t) ** 2).max()), 4))
print("ReLU 導関数の値域        =", sorted(set(np.unique((t > 0).astype(float)).tolist())))
print("GELU の最小値            =", round(float(gelu.min()), 4),
      " (t =", round(float(t[gelu.argmin()]), 2), ")")
print("GELU 導関数の最小値      =", round(float(dgelu.min()), 4))
print("t=4 での導関数: シグモイド", f"{float(sig[-1] * (1 - sig[-1])):.4f}",
      " tanh", f"{float(1 - np.tanh(t[-1]) ** 2):.4f}", " ReLU 1.0000",
      "（飽和するかどうかが勾配の伝わり方を分ける）")

シグモイドの導関数は $t=0$ で最大 $0.2500$、$t=4$ では $0.0177$ まで落ちる。
$\tanh$ の導関数の最大は $1.0000$ だが $t=4$ で $0.0013$ とやはり飽和する。
ReLU の導関数は $\{0,1\}$ の階段関数で正の側では減衰しない。
GELU は $t=-0.75$ で最小値 $-0.1700$ をとる非単調な関数で、導関数の最小値も $-0.1289$ と負になる。
この飽和の有無が深い網での勾配消失（10.8 節）を左右する。

## 10.2 万能近似定理の構成的証明 ― hat 関数と 1 の分割

構成的証明（10.4.2 節）の第 2 段を実装する。$h=1/N$、$t_k=kh$ として hat 関数

$$\Lambda_k(x)=\max\Bigl(0,\,1-\frac{|x-t_k|}{h}\Bigr)\qquad(\text{式 }10.13)$$

は $[0,1]$ 上で 1 の分割 $\sum_{k=0}^N\Lambda_k\equiv1$（式 10.14）をなし、
区分線形補間 $g_N=\sum_k g(t_k)\Lambda_k$（式 10.15）は
$\sup|g-g_N|\le\omega_g(h)$（式 10.16）を満たす。

これを ReLU で書き直す。$\rho(s)=s_+-(s-1)_+$（ランプ関数は ReLU で厳密）を使った
telescope 表現（式 10.17）と、傾きの差分を係数に載せる表現（式 10.23）の 2 通りを実装し、
どちらも `np.interp` の折れ線と機械精度で一致することを確かめる。
不連続な階段関数を経由する素朴な構成では一様近似ができない（10.4.2 節の警告）ことも比較する。

In [ ]:
def hat(x, k, N):
    """式 10.13 の hat 関数 Lambda_k（h = 1/N, t_k = k h）。"""
    h = 1.0 / N
    return np.maximum(0.0, 1.0 - np.abs(x - k * h) / h)


def relu(z):
    return np.maximum(z, 0.0)


def pl_interp_hat(x, g, N):
    """式 10.15 の g_N = sum_k g(t_k) Lambda_k。"""
    tk = np.arange(N + 1) / N
    return sum(g(tk[k]) * hat(x, k, N) for k in range(N + 1))


def pl_interp_ramp(x, g, N):
    """式 10.17（telescope）を ReLU で。ランプ rho(s) = s_+ - (s-1)_+ を 2 個の ReLU で作る。"""
    h, tk = 1.0 / N, np.arange(N + 1) / N
    out = np.full_like(x, g(tk[0]))
    for k in range(1, N + 1):
        s = (x - tk[k - 1]) / h
        out = out + (g(tk[k]) - g(tk[k - 1])) * (relu(s) - relu(s - 1.0))
    return out                      # ReLU 素子は 2N 個


def pl_interp_kink(x, g, N):
    """式 10.23（傾きの差分を ReLU の係数に載せる）。ReLU 素子は N 個。"""
    tk = np.arange(N + 1) / N
    slopes = N * (g(tk[1:]) - g(tk[:-1]))            # 区間 (t_k, t_{k+1}) の傾き s_k
    out = g(tk[0]) + slopes[0] * (x - tk[0])
    for k in range(1, N):
        out = out + (slopes[k] - slopes[k - 1]) * relu(x - tk[k])
    return out


g_fun = lambda x: np.sin(2 * np.pi * x)
xx = np.linspace(0, 1, 20001)
N = 8
y_hat = pl_interp_hat(xx, g_fun, N)
y_ramp = pl_interp_ramp(xx, g_fun, N)
y_kink = pl_interp_kink(xx, g_fun, N)
y_ref = np.interp(xx, np.arange(N + 1) / N, g_fun(np.arange(N + 1) / N))

print("1 の分割 |sum_k Lambda_k - 1| の最大 =",
      f"{np.abs(sum(hat(xx, k, N) for k in range(N + 1)) - 1).max():.2e}")
print("hat 表現 (式 10.15) と np.interp の差 =", f"{np.abs(y_hat - y_ref).max():.2e}")
print("ramp 表現 (式 10.17, ReLU 素子 %d 個) との差 =" % (2 * N),
      f"{np.abs(y_ramp - y_ref).max():.2e}")
print("kink 表現 (式 10.23, ReLU 素子 %d 個) との差 =" % N,
      f"{np.abs(y_kink - y_ref).max():.2e}")
print("節点での補間条件 max|g_N(t_k) - g(t_k)| =",
      f"{np.abs(pl_interp_hat(np.arange(N + 1) / N, g_fun, N) - g_fun(np.arange(N + 1) / N)).max():.2e}")

In [ ]:
from scipy.special import expit          # 数値的に安定なロジスティック関数


# 階段（バンプ）構成との比較：連続な hat か、不連続な階段か
def bump_approx(x, g, N, a):
    """二つのシグモイドの差（式 10.10）を並べた階段近似。シグモイド素子は 2N 個。"""
    tk = np.arange(N + 1) / N
    mid = 0.5 * (tk[:-1] + tk[1:])
    out = np.zeros_like(x)
    for k in range(N):
        beta = expit(a * (x - tk[k])) - expit(a * (x - tk[k + 1]))   # 式 10.10
        out = out + g(mid[k]) * beta
    return out


# 講義ノートの例「1 次元での数値実験」と同じ目標関数。g(1) = 1/2 != 0 なので
# 階段構成が区間端で取りこぼす分がそのまま一様誤差として残る。
g_b = lambda x: np.sin(2 * np.pi * x) + x / 2


fig, axes = plt.subplots(1, 3, figsize=(14, 4.0))
for k in range(N + 1):
    axes[0].plot(xx, hat(xx, k, N), lw=1.2,
                 color=C["blue"] if k % 2 == 0 else C["orange"])
axes[0].plot(xx, sum(hat(xx, k, N) for k in range(N + 1)), "k--", lw=1.4,
             label=L("総和 = 1（1 の分割）", "sum = 1 (partition of unity)"))
axes[0].set_title(L("hat 関数 $\\Lambda_k$（$N=8$）", "hat functions ($N=8$)"))
axes[0].set_xlabel("$x$"); axes[0].legend(fontsize=9)

axes[1].plot(xx, g_b(xx), color="k", lw=1.6, label="$g(x)=\\sin 2\\pi x + x/2$")
for Nv, col in [(4, C["red"]), (8, C["orange"]), (16, C["green"])]:
    axes[1].plot(xx, pl_interp_kink(xx, g_b, Nv), color=col, lw=1.3,
                 label=L("$g_N$（$N=%d$）" % Nv, "$g_N$ ($N=%d$)" % Nv))
axes[1].set_title(L("hat 構成（ReLU で厳密）", "hat construction (exact with ReLU)"))
axes[1].set_xlabel("$x$"); axes[1].legend(fontsize=9)

axes[2].plot(xx, g_b(xx), color="k", lw=1.6)
for Nv, col in [(4, C["red"]), (8, C["orange"]), (16, C["green"])]:
    axes[2].plot(xx, bump_approx(xx, g_b, Nv, 50 * Nv), color=col, lw=1.3,
                 label=L("$N=%d$" % Nv, "$N=%d$" % Nv))
axes[2].set_title(L("階段（バンプ）構成", "step (bump) construction"))
axes[2].set_xlabel("$x$"); axes[2].legend(fontsize=9)
plt.show()

print(f"{'N':>5}{'hat 構成の一様誤差':>20}{'バンプ構成の一様誤差':>22}{'バンプの L2 誤差':>18}")
for Nv in [8, 16, 32, 64, 128]:
    e_hat = float(np.abs(g_b(xx) - pl_interp_kink(xx, g_b, Nv)).max())
    yb = bump_approx(xx, g_b, Nv, 50 * Nv)
    e_b = float(np.abs(g_b(xx) - yb).max())
    l2 = float(np.sqrt(np.trapezoid((g_b(xx) - yb) ** 2, xx)))
    print(f"{Nv:5d}{e_hat:20.2e}{e_b:22.4f}{l2:18.4f}")

3 つの実装（hat、ランプの telescope、傾き差分）はすべて `np.interp` の折れ線と
$10^{-15}$ 程度で一致する。すなわち区分線形補間 $g_N$ は ReLU の 1 隠れ層で**厳密に**表現できる
（命題「ReLU 網は区分線形関数を厳密に表現する」）。式 10.17 では $2N$ 素子、
式 10.23 では $N$ 素子で足りる。1 の分割（式 10.14）の残差は $0$、節点での補間条件も厳密に成立する。

最後の表が 10.4.2 節の警告の数値版である（目標関数は講義ノートの例と同じ
$g(x)=\sin2\pi x+x/2$、バンプの傾きは $a=50N$）。
hat 構成の一様誤差は $N=8,\dots,128$ で
$7.04\times10^{-2},1.88\times10^{-2},4.79\times10^{-3},1.20\times10^{-3},3.01\times10^{-4}$ と減るのに対し、
バンプ（階段）構成の一様誤差は $0.4570,0.3554,0.3029,0.2765,0.2632$ で $0$ に収束しない。
一方その $L^2$ 誤差は $0.1467\to0.0094$ と $O(1/N)$ で減る。
**ノルムの選び方が結論を変える**わけで、不連続な階段関数は連続関数の一様極限になりえない、
という一行がこの差の理由である（$L^2$ 版は 10.4.2 節の注意のとおり正当化できる）。

## 10.3 近似誤差の収束次数 $O(N^{-2})$

定理「1 次元 $C^2$ 関数の近似率」は $\|g-f\|_\infty\le M/(8m^2)$（$M=\max|g''|$）を主張する。
$g(x)=\sin(2\pi x)$ では $M=4\pi^2$ なので上界の定数は $M/8=4.9348$ である。
$N$ 等分の折れ線の一様誤差を測り、$N^2\times$ 誤差が $M/8$ に近づくこと、
両対数プロットの傾きが $-2$ になることを確かめる。

粗い分割（$N=4$）は 1 周期を 4 分割するだけで式 10.24 の
「区間内で $g''$ がほぼ一定」という描像が効かないので、**回帰からは除く**。

In [ ]:
Ns = np.array([4, 8, 16, 32, 64, 128, 256])
errs = np.array([float(np.abs(g_fun(xx) - pl_interp_kink(xx, g_fun, int(Nv))).max())
                 for Nv in Ns])
M2 = 4 * np.pi ** 2

print(f"{'N':>6}{'一様誤差':>14}{'N^2 x 誤差':>14}")
for Nv, e in zip(Ns, errs):
    print(f"{Nv:6d}{e:14.3e}{Nv ** 2 * e:14.3f}")
print("理論上界の定数 M/8 =", round(M2 / 8, 4), " (M = max|g''| = 4 pi^2)")
print("上界を満たすか:", bool(np.all(errs <= M2 / (8 * Ns ** 2) + 1e-12)))

slope_all = np.polyfit(np.log(Ns), np.log(errs), 1)[0]
mask = Ns >= 16
slope_fine = np.polyfit(np.log(Ns[mask]), np.log(errs[mask]), 1)[0]
print("両対数の傾き（全点）      =", round(float(slope_all), 3))
print("両対数の傾き（N >= 16）   =", round(float(slope_fine), 3), " ← 理論値 -2")

fig, ax = plt.subplots(figsize=(6.4, 4.4))
ax.loglog(Ns, errs, "o-", color=C["blue"], label=L("実測（区分線形補間）", "measured"))
ax.loglog(Ns, M2 / (8 * Ns ** 2), "--", color=C["red"],
          label=L("理論上界 $M/(8N^2)$", "bound $M/(8N^2)$"))
ax.set_xlabel("$N$"); ax.set_ylabel(L("一様誤差", "uniform error"))
ax.set_title(L("誤差の収束次数（傾き %.2f）" % slope_fine,
               "convergence rate (slope %.2f)" % slope_fine))
ax.legend(fontsize=9)
plt.show()

一様誤差は $N=4$ の $2.105\times10^{-1}$ から $N=256$ の $7.53\times10^{-5}$ まで落ち、
$N^2\times$ 誤差は $3.368,4.504,4.825,4.907,4.928,4.933,4.934$ と定数に収束する。
理論上界の定数 $M/8=4.9348$ にほぼ一致するので、この評価はタイトである。
両対数の傾きは全点だと $-1.931$、粗い $N=4,8$ を除く $N\ge16$ では $-1.993$ で、
定理の $O(N^{-2})$ が数値的に確認できた。

## 10.4 折れ目の数と深さの効果

命題「ReLU 網は区分線形関数を厳密に表現する」より、幅 $m$ の 1 隠れ層 ReLU 網の折れ目は
**高々 $m$ 個**である。一方、三角波 $T(x)=1-|2x-1|$（幅 2 の 1 隠れ層で厳密）を $L$ 回合成すると
折れ目は $2^L-1$ 個に**指数的に**増える（定理「三角波の合成は折れ点を指数的に増やす」）。
パラメータ数は $L$ に比例するだけである。図 10.4 に対応する。

In [ ]:
def count_kinks(x, y, tol=1e-8):
    """細かい格子上の折れ目（傾きが変わる点）の個数。折れ目が格子点の間にあると
    傾きの変化が 2 回に分かれるので、隣り合う検出は 1 個にまとめる。"""
    sl = np.diff(y) / np.diff(x)
    idx = np.flatnonzero(np.abs(np.diff(sl)) > tol * max(1.0, float(np.abs(sl).max())))
    return 0 if idx.size == 0 else int(1 + np.sum(np.diff(idx) > 1))


rng = np.random.default_rng(0)
xg = np.linspace(0, 1, 200001)
print(f"{'幅 m':>6}{'区間内の折れ目候補':>20}{'実測の折れ目数':>16}")
for m in [2, 4, 8, 16, 32]:
    w = rng.normal(0, 2.0, m)
    bb = rng.normal(0, 1.0, m)
    c = rng.normal(0, 1.0, m)
    f_m = (c[:, None] * relu(w[:, None] * xg[None, :] + bb[:, None])).sum(0)
    inside = int(np.sum((-bb / w > 0) & (-bb / w < 1)))
    print(f"{m:6d}{inside:20d}{count_kinks(xg, f_m):16d}")

T = lambda x: 2 * relu(x) - 4 * relu(x - 0.5)     # [0,1] 上で 1 - |2x - 1|（幅 2 の 1 隠れ層）
print("\n" + f"{'L':>4}{'折れ目数':>10}{'2^L - 1':>10}{'パラメータ数 O(L)':>18}")
xl = np.linspace(0, 1, 400001)
for Lc in range(1, 7):
    z = xl.copy()
    for _ in range(Lc):
        z = T(z)
    print(f"{Lc:4d}{count_kinks(xl, z):10d}{2 ** Lc - 1:10d}{4 * Lc + 1:18d}")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
xs = np.linspace(0, 1, 4001)
for Lc, col in [(1, C["blue"]), (2, C["green"]), (3, C["orange"])]:
    z = xs.copy()
    for _ in range(Lc):
        z = T(z)
    axes[0].plot(xs, z, color=col, lw=1.4, label=r"$T^{\circ %d}$" % Lc)
axes[0].set_xlabel("$x$"); axes[0].set_title(L("三角波の合成", "compositions of the sawtooth"))
axes[0].legend(fontsize=9)

Ls = np.arange(1, 11)
axes[1].semilogy(Ls, 2.0 ** Ls - 1, "o-", color=C["red"],
                 label=L("深さ $L$・幅 2 の折れ目数 $2^L-1$", "depth $L$: $2^L-1$ kinks"))
axes[1].semilogy(Ls, 4 * Ls + 1, "s--", color=C["blue"],
                 label=L("そのパラメータ数 $O(L)$", "its parameter count $O(L)$"))
axes[1].semilogy(Ls, (4 * Ls + 1) / 3, "^:", color=C["gray"],
                 label=L("同予算の 1 隠れ層の折れ目数（高々 $m$）",
                         "1 hidden layer with same budget"))
axes[1].set_xlabel("$L$"); axes[1].set_ylabel(L("個数", "count"))
axes[1].set_title(L("深さと幅のトレードオフ", "depth vs width"))
axes[1].legend(fontsize=8)
plt.show()

幅 $m$ の 1 隠れ層 ReLU 網の折れ目は、区間 $(0,1)$ に落ちる $-b_j/w_j$ の個数にちょうど一致し、
$m$ を超えない。一方 $T^{\circ L}$ の折れ目は $L=1,\dots,6$ で $1,3,7,15,31,63$、
すなわち $2^L-1$ である。パラメータ数は $4L+1$ と $L$ に比例するだけなので、
同じ予算の 1 隠れ層（折れ目は高々 $m\approx(4L+1)/3$ 個）では追いつけない。
補題「区分線形片の数の上界」から、隠れ層 $k$ 層・幅 $w$ で $T^{\circ L}$ を表すには
$(2w)^k\ge2^L-1$、すなわち $w=\Omega(2^{L/k})$ が必要である。

## 10.5 誤差逆伝播法のスクラッチ実装と数値微分による検証

命題「誤差逆伝播」（式 10.33）

$$\boldsymbol\delta^{(l)}=\Bigl(\bigl(\boldsymbol{W}^{(l+1)}\bigr)^\top\boldsymbol\delta^{(l+1)}\Bigr)\odot\sigma'\bigl(\boldsymbol{z}^{(l)}\bigr),
\qquad \frac{\partial\ell}{\partial\boldsymbol{W}^{(l)}}=\boldsymbol\delta^{(l)}\bigl(\boldsymbol{h}^{(l-1)}\bigr)^\top$$

を任意の層数について実装し、中心差分（$h=10^{-6}$）と突き合わせる。
配列はすべて $d\times n$（列がサンプル）である。

In [ ]:
def mlp_loss_grad(params, X, Y, act="tanh"):
    """任意層数の MLP の二乗損失と勾配（式 10.32、式 10.33）。X: d×n, Y: n_out×n。"""
    n = X.shape[1]
    Ws = params[0::2]
    bs = params[1::2]
    Lnum = len(Ws)
    H = [X]
    Zs = []
    for l in range(Lnum):
        Z = Ws[l] @ H[-1] + bs[l][:, None]
        Zs.append(Z)
        if l < Lnum - 1:                       # 最終層は線形出力
            H.append(np.tanh(Z) if act == "tanh" else np.maximum(Z, 0.0))
    R = Zs[-1] - Y
    loss = 0.5 * np.sum(R ** 2) / n
    grads = [None] * (2 * Lnum)
    D = R / n                                  # delta^{(L)}
    for l in range(Lnum - 1, -1, -1):
        grads[2 * l] = D @ H[l].T              # dL/dW^{(l)} = delta (h^{(l-1)})^T
        grads[2 * l + 1] = D.sum(1)            # dL/db^{(l)}
        if l > 0:
            dact = (1 - H[l] ** 2) if act == "tanh" else (Zs[l - 1] > 0).astype(float)
            D = (Ws[l].T @ D) * dact           # 式 10.33
    return loss, grads


def numerical_grad(params, X, Y, act="tanh", h=1e-6):
    out = []
    for P in params:
        gn = np.zeros_like(P)
        for idx in np.ndindex(P.shape):
            old = P[idx]
            P[idx] = old + h; lp, _ = mlp_loss_grad(params, X, Y, act)
            P[idx] = old - h; lm, _ = mlp_loss_grad(params, X, Y, act)
            P[idx] = old
            gn[idx] = (lp - lm) / (2 * h)
        out.append(gn)
    return out


rng = np.random.default_rng(0)
d_in, n_s, n_out = 3, 7, 2
Xn = rng.normal(size=(d_in, n_s))               # d×n : 列がサンプル
Yn = rng.normal(size=(n_out, n_s))
sizes = [d_in, 5, 4, n_out]                     # 隠れ 2 層（幅 5, 4）
par = []
for a_, b_ in zip(sizes[:-1], sizes[1:]):
    par += [rng.normal(0, np.sqrt(2 / a_), (b_, a_)), np.zeros(b_)]   # He 初期化

for act in ["tanh", "relu"]:
    loss, gr = mlp_loss_grad(par, Xn, Yn, act)
    gn = numerical_grad(par, Xn, Yn, act)
    rel = [float(np.max(np.abs(a - b)) / max(np.max(np.abs(b)), 1e-12))
           for a, b in zip(gr, gn)]
    print(f"{act:>5}: 損失 = {loss:.6f}, ブロックごとの最大相対誤差 =",
          " ".join(f"{r:.1e}" for r in rel))
print("勾配の計算コストは順伝播の定数倍（数値微分はパラメータ数",
      sum(P.size for P in par), "回の順伝播が必要）")

`tanh` では 6 つのパラメータブロック（$\boldsymbol{W}^{(l)},\boldsymbol{b}^{(l)}$ が 3 層分）すべてで
相対誤差が $2\times10^{-10}$ 前後、
中心差分の丸め誤差の水準で一致する。`relu` でも同程度だが、
$z$ が $0$ の近くにある素子があると片側だけ活性が切り替わって差分が合わなくなるため、
ReLU の勾配検査では最大値ではなく中央値を見るのが安全である
（講義ノートの演習「誤差逆伝播の実装」(2)）。

パラメータ数は 54 で、数値微分には 1 回の勾配につき $2\times54$ 回の順伝播が要る。
逆伝播はこれを順伝播の定数倍で済ませる。

## 10.6 XOR ― 1 隠れ層 2 ユニットで解ける

命題「XOR の表現不可能性」はアフィン関数 $g$ について
$g(\boldsymbol{x}_2)+g(\boldsymbol{x}_3)=g(\boldsymbol{x}_1)+g(\boldsymbol{x}_4)$ が恒等的に成り立つことから従う。
一方、式 10.2 の重み（$\boldsymbol{w}_1=\boldsymbol{w}_2=(1,1)^\top$、$b_1=-1/2$、$b_2=-3/2$、$c=(1,-1)$）は
XOR を厳密に表現する。滑らかなシグモイドでも傾き $a=40$ で機械精度に達する。

In [ ]:
Xx = np.array([[0., 0., 1., 1.],
               [0., 1., 0., 1.]])            # d×B = 2×4 : 列が入力
yx = np.array([0., 1., 1., 0.])              # XOR

Wx = np.array([[1., 1.], [1., 1.]])          # m×d, w_1 = w_2 = (1,1)
bx = np.array([-0.5, -1.5])                  # 閾値 1/2 と 3/2
cx = np.array([1.0, -1.0])                   # 出力側の重み（差をとる）

H_step = (Wx @ Xx + bx[:, None] > 0).astype(float)      # 階段関数の場合
H_sig = 1 / (1 + np.exp(-40.0 * (Wx @ Xx + bx[:, None])))   # a = 40 のシグモイド
print("階段活性の出力      :", (cx @ H_step).round(6))
print("シグモイド(a=40) 出力:", (cx @ H_sig).round(6))
print("目標 (XOR)          :", yx)
print("シグモイド版の最大誤差:", f"{np.abs(cx @ H_sig - yx).max():.2e}")

# 単一のアフィン関数では不可能：g(x2)+g(x3) - g(x1) - g(x4) は恒等的に 0
rng = np.random.default_rng(1)
gaps = []
for _ in range(1000):
    wv, bv = rng.normal(size=2), rng.normal()
    gv = wv @ Xx + bv
    gaps.append(gv[1] + gv[2] - gv[0] - gv[3])
print("ランダムなアフィン関数 1000 個での |g(x2)+g(x3)-g(x1)-g(x4)| の最大 =",
      f"{np.abs(gaps).max():.1e}",
      "→ ラベル 1 の 2 点とラベル 0 の 2 点の中点が一致するので線形分離できない")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
axes[0].scatter(Xx[0, yx > 0], Xx[1, yx > 0], s=120, color=C["red"], label="$y=1$")
axes[0].scatter(Xx[0, yx == 0], Xx[1, yx == 0], s=120, color=C["blue"], marker="s",
                label="$y=0$")
gx = np.linspace(-0.4, 1.4, 50)
for bb_, col in [(-0.5, C["green"]), (-1.5, C["orange"])]:
    axes[0].plot(gx, -gx - bb_, color=col, lw=1.5,
                 label=L("隠れ素子 $x_1+x_2=%g$" % (-bb_), "hidden unit $x_1+x_2=%g$" % (-bb_)))
axes[0].set_xlim(-0.4, 1.4); axes[0].set_ylim(-0.4, 1.4)
axes[0].set_xlabel("$x_1$"); axes[0].set_ylabel("$x_2$")
axes[0].set_title(L("入力空間：1 本の直線では分けられない", "input space: not separable"))
axes[0].legend(fontsize=8)

axes[1].scatter(H_sig[0, yx > 0], H_sig[1, yx > 0], s=120, color=C["red"], label="$y=1$")
axes[1].scatter(H_sig[0, yx == 0], H_sig[1, yx == 0], s=120, color=C["blue"], marker="s",
                label="$y=0$")
hh = np.linspace(-0.1, 1.1, 50)
axes[1].plot(hh, hh - 0.5, "k--", lw=1.4, label=L("$h_1-h_2=1/2$", "$h_1-h_2=1/2$"))
axes[1].set_xlabel("$h_1$"); axes[1].set_ylabel("$h_2$")
axes[1].set_title(L("隠れ層の出力：線形分離できる", "hidden layer: separable"))
axes[1].legend(fontsize=8)
plt.show()

階段活性では出力がちょうど $(0,1,1,0)$、$a=40$ のシグモイドでも誤差 $4.1\times10^{-9}$ で
XOR を表現する。ランダムなアフィン関数 1000 個で
$|g(\boldsymbol{x}_2)+g(\boldsymbol{x}_3)-g(\boldsymbol{x}_1)-g(\boldsymbol{x}_4)|$ の最大は $8.9\times10^{-16}$、
つまり恒等的に $0$ であり、ラベル $1$ の 2 点の中点とラベル $0$ の 2 点の中点が一致する。
これが線形分離できない理由である。
隠れ層は 4 点を $s=x_1+x_2$ 軸上の 3 点に潰し、中央の帯を切り出す。
右図のとおり隠れ層の出力 $(h_1,h_2)$ では 1 本の直線で分けられる。

## 10.7 学習の実験と Barron クラス

ここまでは「表現できる」話だった。最後に**学習できるか**を見る。
`sklearn.neural_network.MLPRegressor`（`tanh`、L-BFGS）で
$g(x)=\sin(2\pi x)+\tfrac12\sin(6\pi x)$ を $[0,1]$ 上の 1000 点から近似し、
幅 $m$ を変えたときの $L^2$ 誤差の減衰を測る。比較の対象は二つある。

- 定理「Barron 1993」（式 10.26）：$L^2$ 誤差 $\le(2rC_f)^2/m$、すなわち $O(m^{-1/2})$。
- 定理「1 次元 $C^2$ 関数の近似率」：最適な折れ線なら $O(m^{-2})$。

$g$ の Fourier 測度は $\pm2\pi$ に質量 $1/2$、$\pm6\pi$ に質量 $1/4$ の原子測度なので
Barron 定数は $C_f=2\pi\cdot1+6\pi\cdot\tfrac12=5\pi=15.708$ である（例「Barron 定数の計算例」と同じ計算）。
深さの比較（ほぼ同じパラメータ数で幅と層数を振り替える）も行う。

In [ ]:
from sklearn.neural_network import MLPRegressor

g2 = lambda x: np.sin(2 * np.pi * x) + 0.5 * np.sin(6 * np.pi * x)
Xtr = np.linspace(0, 1, 1000)[None, :]        # d×n = 1×1000（列がサンプル）
ytr = g2(Xtr[0])


def fit_mlp(hidden, seeds=(0, 1, 2, 3, 4)):
    """L2 誤差（訓練点上の RMSE）の中央値を返す。sklearn は n×d 規約なので転置して渡す。"""
    es = []
    for s in seeds:
        net = MLPRegressor(hidden_layer_sizes=hidden, activation="tanh", solver="lbfgs",
                           alpha=1e-8, max_iter=4000, tol=1e-10, random_state=s)
        net.fit(Xtr.T, ytr)
        es.append(float(np.sqrt(np.mean((net.predict(Xtr.T) - ytr) ** 2))))
    return float(np.median(es)), es


widths = [2, 4, 8, 16, 32, 64]
med = []
print(f"{'幅 m':>6}{'L2 誤差(中央値)':>18}{'最小':>12}{'最大':>12}")
for m in widths:
    mm, es = fit_mlp((m,))
    med.append(mm)
    print(f"{m:6d}{mm:18.5f}{min(es):12.5f}{max(es):12.5f}")

med = np.array(med)
slope = float(np.polyfit(np.log(widths), np.log(med), 1)[0])
Cf = 5 * np.pi                                  # Barron 定数（原子測度の計算）
bound = (2 * 1.0 * Cf) ** 2 / np.array(widths)  # 式 10.26（r = 1）
print("\n両対数の傾き =", round(slope, 3),
      " （Barron の $O(m^{-1/2})$ は -0.5、折れ線の $O(m^{-2})$ は -2）")
print("Barron 定数 C_f = 5 pi =", round(Cf, 4))
print(f"{'幅 m':>6}{'実測の二乗誤差':>16}{'Barron 上界 (2rC_f)^2/m':>26}")
for m, e, bd in zip(widths, med, bound):
    print(f"{m:6d}{e ** 2:16.2e}{bd:26.1f}")

In [ ]:
budgets = [((32,), "1 隠れ層 (32)"), ((8, 8), "2 隠れ層 (8,8)"), ((6, 6, 6), "3 隠れ層 (6,6,6)")]
print(f"{'構成':<22}{'パラメータ数':>12}{'中央値':>10}{'最小':>10}{'最大':>10}")
for hid, name in budgets:
    sizes_ = [1] + list(hid) + [1]
    npar = sum(a * b + b for a, b in zip(sizes_[:-1], sizes_[1:]))
    mm, es = fit_mlp(hid)
    print(f"{name:<22}{npar:12d}{mm:10.5f}{min(es):10.5f}{max(es):10.5f}")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3))
axes[0].loglog(widths, med, "o-", color=C["blue"], label=L("実測（学習後）", "measured"))
ref = med[0] * (np.array(widths) / widths[0]) ** -0.5
axes[0].loglog(widths, ref, "--", color=C["red"], label="$O(m^{-1/2})$")
axes[0].loglog(widths, med[0] * (np.array(widths) / widths[0]) ** -2.0, ":",
               color=C["green"], label="$O(m^{-2})$")
axes[0].set_xlabel("$m$"); axes[0].set_ylabel(L("$L^2$ 誤差", "$L^2$ error"))
axes[0].set_title(L("幅と近似誤差（傾き %.2f）" % slope, "width vs error (slope %.2f)" % slope))
axes[0].legend(fontsize=9)

net = MLPRegressor(hidden_layer_sizes=(32,), activation="tanh", solver="lbfgs",
                   alpha=1e-8, max_iter=4000, tol=1e-10, random_state=0).fit(Xtr.T, ytr)
xs2 = np.linspace(0, 1, 1000)
axes[1].plot(xs2, g2(xs2), color="k", lw=1.6, label="$g(x)$")
axes[1].plot(xs2, net.predict(xs2[:, None]), color=C["orange"], lw=1.4,
             label=L("学習した幅 32 の網", "trained width-32 net"))
axes[1].set_xlabel("$x$"); axes[1].set_title(L("学習結果（幅 32）", "fit with width 32"))
axes[1].legend(fontsize=9)
plt.show()

学習後の $L^2$ 誤差（5 個の乱数初期値の中央値）は幅 $m=2$ の $0.2216$ から
$m=64$ の $0.00785$ まで落ち、両対数の傾きは $-1.069$ であった。
Barron の $O(m^{-1/2})$ より速く、最適な折れ線の $O(m^{-2})$ よりは遅い。
前者は最悪ケースの上界なので緩い：$m=64$ での実測の二乗誤差 $6.2\times10^{-5}$ に対し
上界 $(2rC_f)^2/m$ は $15.4$ で、5 桁以上の開きがある。
後者に届かないのは、勾配法が最適な折れ点配置に到達するとは限らないためである。
定理は近似の**存在**を保証するだけで、最適化がそこに届くことは保証しない（10.4.3 節の限界 (2)）。
$m=8$ の最大値 $0.158$ のように、初期値によっては悪い解に落ちることも表から読み取れる。

同じパラメータ予算（97, 97, 103）で幅と深さを振り替えると、中央値は
1 隠れ層 (32) が $0.00924$、2 隠れ層 (8,8) が $0.00560$、3 隠れ層 (6,6,6) が $0.00489$ で、
深いほうがわずかに小さい。ただし (8,8) の最大値は $0.127$ と中央値の 20 倍以上あり、
**初期値によるばらつきが構成間の差より大きい**。
この 1 次元の滑らかな目標関数で深さの優位を結論づけることはできない。
深さが確実に効くのは 10.6 節の $T^{\circ L}$ のように振動が本質的な関数に対してである。

## 演習

**演習 10-1（1 の分割と補間の一意性）**
$N=5$、$g(t)=\sin 3t+0.4t^2$ で (a) $\Lambda_k(t_j)=\delta_{jk}$、
(b) 各小区間で $g_N$ が 1 次式であること（二階差分が $0$）、
(c) 明示形 $g_N(x)=g(t_k)(t_{k+1}-x)/h+g(t_{k+1})(x-t_k)/h$ との一致、
を数値で確かめよ（講義ノートの演習「区分線形補間」）。

**演習 10-2（三角波を $\mathbb{R}$ 全体で表す）**
$T(x)=2x_+-4(x-1/2)_+$ は $x>1$ で負に発散する。第 3 項を足した
$2x_+-4(x-1/2)_++2(x-1)_+$ が $\mathbb{R}$ 全体で $\max(0,1-|2x-1|)$ に一致することを確かめよ。

**演習 10-3（勾配消失と初期化）**
幅 100 の ReLU 網を 10 層積み、$v=1/n_{l-1}$（Xavier 型）と $v=2/n_{l-1}$（He 型、式 10.34）で
初期化したときの活性の二乗モーメント $\mathbb{E}[(h^{(l)})^2]$ を層ごとに測り、
He 初期化なら層をまたいで保たれることを確かめよ。

In [ ]:
# 演習 10-1
N5 = 5
tk5 = np.arange(N5 + 1) / N5
g5 = lambda t: np.sin(3 * t) + 0.4 * t ** 2
# TODO: Lam[j,k] = hat(tk5[j], k, N5) を作り、単位行列との差の最大を印字する。
# TODO: 小区間ごとに細かい格子をとり、pl_interp_hat の二階差分の最大を印字する。
# TODO: 明示形との差の最大を印字する。

# 演習 10-2
xr = np.linspace(-0.5, 1.5, 2001)
# TODO: T2 = 2*relu(xr) - 4*relu(xr-0.5) と T3 = T2 + 2*relu(xr-1.0) を作り、
#       np.maximum(0, 1 - np.abs(2*xr - 1)) との最大差をそれぞれ印字する。

# 演習 10-3
# TODO: rng を固定し、h = 標準正規の (100, 4096) から始めて 10 層
#       h = relu(W @ h) （W は N(0, v) で v = 1/100 と 2/100）を回し、
#       各層で (h**2).mean() を記録して片対数プロットする。

## 演習の解答

In [ ]:
# --- 演習 10-1 ---------------------------------------------------------------
Lam = np.array([[hat(np.array([tk5[j]]), k, N5)[0] for k in range(N5 + 1)]
                for j in range(N5 + 1)])
print("(a) max |Lambda_k(t_j) - delta_jk| =",
      f"{np.abs(Lam - np.eye(N5 + 1)).max():.2e}")

worst_2nd, worst_expl = 0.0, 0.0
for k in range(N5):
    xs_k = np.linspace(tk5[k], tk5[k + 1], 201)
    yk_ = pl_interp_hat(xs_k, g5, N5)
    worst_2nd = max(worst_2nd, float(np.abs(np.diff(yk_, 2)).max()))
    h5 = 1.0 / N5
    expl = g5(tk5[k]) * (tk5[k + 1] - xs_k) / h5 + g5(tk5[k + 1]) * (xs_k - tk5[k]) / h5
    worst_expl = max(worst_expl, float(np.abs(yk_ - expl).max()))
print("(b) 各小区間での二階差分の最大 =", f"{worst_2nd:.2e}", "（1 次式なので 0）")
print("(c) 明示形との差の最大         =", f"{worst_expl:.2e}")

# --- 演習 10-2 ---------------------------------------------------------------
T2 = 2 * relu(xr) - 4 * relu(xr - 0.5)
T3 = T2 + 2 * relu(xr - 1.0)
tgt = np.maximum(0.0, 1.0 - np.abs(2 * xr - 1))
print("\n2 項表現との最大差 =", f"{np.abs(T2 - tgt).max():.4f}",
      "（x > 1 で負に発散する）")
print("3 項表現との最大差 =", f"{np.abs(T3 - tgt).max():.2e}",
      "（R 全体で一致）")

# --- 演習 10-3 ---------------------------------------------------------------
rng_e = np.random.default_rng(0)
width, nsamp, depth = 100, 4096, 10
h0 = rng_e.normal(size=(width, nsamp))          # 幅×サンプル（列がサンプル）
curves = {}
for name, v in [("Xavier 型 v=1/n", 1.0 / width), ("He 型 v=2/n", 2.0 / width)]:
    h_ = h0.copy()
    mom = [float((h_ ** 2).mean())]
    for _ in range(depth):
        W_ = rng_e.normal(0, np.sqrt(v), (width, width))
        h_ = relu(W_ @ h_)
        mom.append(float((h_ ** 2).mean()))
    curves[name] = mom
    print(f"\n{name}: E[h^2] =", " ".join(f"{m:.3f}" for m in mom))

fig, ax = plt.subplots(figsize=(6.6, 4.2))
for (name, mom), col in zip(curves.items(), [C["blue"], C["red"]]):
    ax.semilogy(range(depth + 1), mom, "o-", color=col, label=name)
ax.set_xlabel(L("層 $l$", "layer $l$")); ax.set_ylabel(r"$\mathbb{E}[(h^{(l)})^2]$")
ax.set_title(L("初期化と活性の二乗モーメント（式 10.34）",
               "initialization and second moment of activations"))
ax.legend(fontsize=9)
plt.show()
print("\n→ He 初期化では E[h^2] が 1 のまわりに保たれ、Xavier 型では層ごとに約半分になる。")